# IoT Device Identification with Adversarial Robustness
## Google Colab Training Notebook

**Implementation of:** "Enhancing SDN-enabled Network Access Control with Adversarial Robust IoT Device Identification"

### Paths:
- **Data:** `/content/drive/MyDrive/PFE/IPFIX_ML_Instances/`
- **Results:** `/content/drive/MyDrive/PFE/results/`

## 0. Setup Environment

In [ ]:
!pip install xgboost -q
!pip install tensorflow -q

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle
import gc

from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

## 1. Mount Google Drive & Setup Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_PATH = '/content/drive/MyDrive/PFE/IPFIX_ML_Instances/'
OUTPUT_PATH = '/content/drive/MyDrive/PFE/results/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

print('Data path:', DATA_PATH)
print('Output path:', OUTPUT_PATH)
print('\nFiles in data directory:')
!ls -lh {DATA_PATH}

## 2. Configuration

In [ ]:
SDN_FEATURES = [
    'duration', 'ipProto', 'outPacketCount', 'outByteCount', 'inPacketCount', 'inByteCount',
    'outSmallPktCount', 'outLargePktCount', 'outNonEmptyPktCount', 'outDataByteCount',
    'outAvgIAT', 'outFirstNonEmptyPktSize', 'outMaxPktSize', 'outStdevPayloadSize',
    'outStdevIAT', 'outAvgPacketSize', 'inSmallPktCount', 'inLargePktCount',
    'inNonEmptyPktCount', 'inDataByteCount', 'inAvgIAT', 'inFirstNonEmptyPktSize',
    'inMaxPktSize', 'inStdevPayloadSize', 'inStdevIAT', 'inAvgPacketSize',
    'http', 'https', 'smb', 'dns', 'ntp', 'tcp', 'udp', 'ssdp', 'lan', 'wan'
]

TARGET = 'name'

DTYPE_DICT = {
    'duration': 'float32',
    'ipProto': 'int16',
    'outPacketCount': 'int32',
    'outByteCount': 'int64',
    'inPacketCount': 'int32',
    'inByteCount': 'int64',
    'outSmallPktCount': 'int32',
    'outLargePktCount': 'int32',
    'outNonEmptyPktCount': 'int32',
    'outDataByteCount': 'int64',
    'outAvgIAT': 'float32',
    'outFirstNonEmptyPktSize': 'int32',
    'outMaxPktSize': 'int32',
    'outStdevPayloadSize': 'float32',
    'outStdevIAT': 'float32',
    'outAvgPacketSize': 'float32',
    'inSmallPktCount': 'int32',
    'inLargePktCount': 'int32',
    'inNonEmptyPktCount': 'int32',
    'inDataByteCount': 'int64',
    'inAvgIAT': 'float32',
    'inFirstNonEmptyPktSize': 'int32',
    'inMaxPktSize': 'int32',
    'inStdevPayloadSize': 'float32',
    'inStdevIAT': 'float32',
    'inAvgPacketSize': 'float32',
    'http': 'int8',
    'https': 'int8',
    'smb': 'int8',
    'dns': 'int8',
    'ntp': 'int8',
    'tcp': 'int8',
    'udp': 'int8',
    'ssdp': 'int8',
    'lan': 'int8',
    'wan': 'int8',
    'device': 'category',
    'name': 'category'
}

SAMPLE_RATIO = 1.0  # Set to < 1.0 to use subset for testing

VALID_CLASSES = [
    'eclear', 'sleep', 'esensor', 'hub-plus', 'humidifier',
    'home-unit', 'inkjet-printer', 'smart-wifi-plug-mini', 'smart-power-strip',
    'echo-dot', 'fire7-tablet', 'google-nest-mini', 'google-chromecast',
    'atom-cam', 'kasa-camera-pro', 'kasa-smart-led-lamp', 'fire-tv-stick-4k', 'qrio-hub'
]

## 3. Data Loading (Memory Optimized)

In [ ]:
def load_csv_optimized(filepath, features, target):
    usecols = [f for f in features if f != target] + [target]
    available = pd.read_csv(filepath, nrows=0).columns.tolist()
    usecols = [c for c in usecols if c in available]
    
    dtype_subset = {k: v for k, v in DTYPE_DICT.items() if k in usecols}
    df = pd.read_csv(filepath, usecols=usecols, dtype=dtype_subset, low_memory=True)
    return df

In [ ]:
def load_all_data(data_path, sample_ratio=1.0):
    all_dfs = []
    
    for i in range(1, 13):
        filepath = os.path.join(data_path, f'home{i}_labeled.csv')
        if os.path.exists(filepath):
            print(f'Loading home{i}_labeled.csv...')
            df = load_csv_optimized(filepath, SDN_FEATURES, TARGET)
            
            if sample_ratio < 1.0:
                df = df.sample(frac=sample_ratio, random_state=42)
            
            mem_mb = df.memory_usage(deep=True).sum() / 1024**2
            print(f'  Shape: {df.shape}, Memory: {mem_mb:.1f} MB')
            all_dfs.append(df)
            gc.collect()
    
    print('\nConcatenating dataframes...')
    df = pd.concat(all_dfs, ignore_index=True)
    del all_dfs
    gc.collect()
    
    return df

In [ ]:
print('='*60)
print('Loading Dataset')
print('='*60)
df = load_all_data(DATA_PATH, sample_ratio=SAMPLE_RATIO)
print(f'\nTotal dataset shape: {df.shape}')
print(f'Total memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

## 4. Data Preprocessing

In [ ]:
def preprocess_data(df, features, target, valid_classes=None):
    print('Preprocessing...')
    
    valid_features = [f for f in features if f in df.columns]
    print(f'Valid features: {len(valid_features)}/{len(features)}')
    
    df = df.dropna(subset=[target]).copy()
    
    if valid_classes:
        before = len(df)
        df = df[df[target].isin(valid_classes)].copy()
        print(f'Filtered to {len(valid_classes)} classes: {before} -> {len(df)} samples')
    
    for col in valid_features:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(0).astype('float32')
    
    df = df.replace([np.inf, -np.inf], 0)
    
    X = df[valid_features].values.astype('float32')
    y = df[target].values
    
    return X, y, valid_features

In [ ]:
X, y, feature_names = preprocess_data(df, SDN_FEATURES, TARGET, valid_classes=VALID_CLASSES)
del df
gc.collect()
print(f'Feature matrix: {X.shape}')

In [ ]:
print('Encoding labels...')
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

print(f'Number of classes: {num_classes}')
print(f'Classes: {list(label_encoder.classes_)}')

with open(os.path.join(OUTPUT_PATH, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(label_encoder, f)

del y
gc.collect()

In [ ]:
print('Scaling features...')
minmax_scaler = MinMaxScaler()
X_minmax = minmax_scaler.fit_transform(X)

standard_scaler = StandardScaler()
X_scaled = standard_scaler.fit_transform(X_minmax)

with open(os.path.join(OUTPUT_PATH, 'scaler.pkl'), 'wb') as f:
    pickle.dump({'minmax': minmax_scaler, 'standard': standard_scaler}, f)

del X, X_minmax
gc.collect()
print(f'Scaled features: {X_scaled.shape}')

In [ ]:
print('Splitting data...')
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.25, random_state=42, stratify=y_encoded
)

print(f'Training: {X_train.shape[0]:,} samples')
print(f'Test: {X_test.shape[0]:,} samples')

## 5. Model Training

In [ ]:
def predict(model, X):
    y_pred = model.predict(X)
    if len(y_pred.shape) > 1:
        y_pred = np.argmax(y_pred, axis=1)
    return y_pred

def evaluate_model(model, X_test, y_test, name):
    y_pred = predict(model, X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    print(f'{name}: F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}')
    return {'model': name, 'f1': f1, 'precision': precision, 'recall': recall}

In [ ]:
models = {}

### 5.1 Random Forest

In [ ]:
print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, criterion='gini', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

with open(os.path.join(OUTPUT_PATH, 'rf_model.pkl'), 'wb') as f:
    pickle.dump(rf, f)

models['RF'] = rf
evaluate_model(rf, X_test, y_test, 'Random Forest')

### 5.2 XGBoost

In [ ]:
print('Training XGBoost...')
xgb_model = xgb.XGBClassifier(
    n_estimators=100, eval_metric='merror', random_state=42, n_jobs=-1, use_label_encoder=False
)
xgb_model.fit(X_train, y_train)

with open(os.path.join(OUTPUT_PATH, 'xgb_model.pkl'), 'wb') as f:
    pickle.dump(xgb_model, f)

models['XGBoost'] = xgb_model
evaluate_model(xgb_model, X_test, y_test, 'XGBoost')

### 5.3 KNN

In [ ]:
print('Training KNN...')
knn = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)
knn.fit(X_train, y_train)

with open(os.path.join(OUTPUT_PATH, 'knn_model.pkl'), 'wb') as f:
    pickle.dump(knn, f)

models['KNN'] = knn
evaluate_model(knn, X_test, y_test, 'KNN')

### 5.4 DNN

In [ ]:
print('Training DNN...')
dnn = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])
dnn.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
dnn.summary()

In [ ]:
history = dnn.fit(X_train, y_train, epochs=30, batch_size=256, validation_split=0.2, verbose=1)
dnn.save(os.path.join(OUTPUT_PATH, 'dnn_model.h5'))
models['DNN'] = dnn
evaluate_model(dnn, X_test, y_test, 'DNN')

## 6. Adversarial Attack Generation

In [ ]:
class AdversarialAttackGenerator:
    def __init__(self, model, num_classes):
        self.model = model
        self.num_classes = num_classes
    
    def predict_proba(self, X):
        if hasattr(self.model, 'predict_proba'):
            return self.model.predict_proba(X)
        else:
            return self.model.predict(X)
    
    def generate_attack(self, x, y, max_iter=50, epsilon=0.05):
        x_adv = x.copy()
        targets = [c for c in range(self.num_classes) if c != y]
        tgt = np.random.choice(targets)
        
        for _ in range(max_iter):
            pred = self.predict_proba(x_adv.reshape(1, -1))
            if len(pred.shape) > 1:
                pred_class = np.argmax(pred[0])
                pred_probs = pred[0]
            else:
                break
            
            if pred_class == tgt:
                break
            
            diff = np.abs(1 - pred_probs[tgt])
            perturb = epsilon * np.sign(np.random.randn(len(x))) * diff
            x_adv = np.clip(x_adv + perturb, -3, 3)
        
        return x_adv

In [ ]:
n_adv = min(1000, len(X_test) // 4)
X_test_sample = X_test[:n_adv]
y_test_sample = y_test[:n_adv]

print(f'Generating {n_adv} adversarial samples...')
attack_gen = AdversarialAttackGenerator(models['XGBoost'], num_classes)

X_adv = []
for i, (x, y) in enumerate(zip(X_test_sample, y_test_sample)):
    x_adv = attack_gen.generate_attack(x, y)
    X_adv.append(x_adv)
    if (i+1) % 100 == 0:
        print(f'  Generated {i+1}/{n_adv}')

X_adv = np.array(X_adv)
print(f'Adversarial samples shape: {X_adv.shape}')

In [ ]:
print('\nAdversarial Impact:')
print('='*50)
impact_results = []
for name, model in models.items():
    f1_clean = f1_score(y_test_sample, predict(model, X_test_sample), average='weighted')
    f1_adv = f1_score(y_test_sample, predict(model, X_adv), average='weighted')
    drop = f1_clean - f1_adv
    print(f'{name}: F1_clean={f1_clean:.4f}, F1_adv={f1_adv:.4f}, Drop={drop:.4f}')
    impact_results.append({'model': name, 'f1_clean': f1_clean, 'f1_adv': f1_adv, 'drop': drop})

pd.DataFrame(impact_results).to_csv(os.path.join(OUTPUT_PATH, 'adversarial_impact.csv'), index=False)

## 7. Defense: Adversarial Detection

In [ ]:
n_det = min(len(X_adv), len(X_train))
X_det = np.vstack([X_train[:n_det], X_adv])
y_det = np.hstack([np.zeros(n_det), np.ones(n_det)])

X_det_train, X_det_test, y_det_train, y_det_test = train_test_split(
    X_det, y_det, test_size=0.25, random_state=42, stratify=y_det
)

print(f'Detection data: {X_det_train.shape[0]} train, {X_det_test.shape[0]} test')

In [ ]:
print('Training Detection RF...')
det_rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
det_rf.fit(X_det_train, y_det_train)

y_pred = predict(det_rf, X_det_test)
f1 = f1_score(y_det_test, y_pred, average='weighted')
print(f'Detection RF F1: {f1:.4f}')

with open(os.path.join(OUTPUT_PATH, 'detection_rf.pkl'), 'wb') as f:
    pickle.dump(det_rf, f)

## 8. Defense: Adversarial Training

In [ ]:
n_robust = min(len(X_adv), len(X_train) // 4)
X_robust = np.vstack([X_train, X_adv[:n_robust]])
y_robust = np.hstack([y_train, y_test_sample[:n_robust]])

print(f'Robust training: {X_robust.shape[0]} samples')

In [ ]:
print('Training Robust RF...')
robust_rf = RandomForestClassifier(n_estimators=200, bootstrap=True, random_state=42, n_jobs=-1)
robust_rf.fit(X_robust, y_robust)

f1_clean = f1_score(y_test_sample, predict(robust_rf, X_test_sample), average='weighted')
f1_adv = f1_score(y_test_sample, predict(robust_rf, X_adv), average='weighted')
recovery = f1_adv / f1_clean * 100

print(f'Robust RF: F1_clean={f1_clean:.4f}, F1_adv={f1_adv:.4f}, Recovery={recovery:.1f}%')

with open(os.path.join(OUTPUT_PATH, 'robust_rf.pkl'), 'wb') as f:
    pickle.dump(robust_rf, f)

## 9. Save Results

In [ ]:
print('\n' + '='*60)
print('FINAL RESULTS')
print('='*60)

results = []
for name, model in models.items():
    r = evaluate_model(model, X_test, y_test, name)
    results.append(r)

pd.DataFrame(results).to_csv(os.path.join(OUTPUT_PATH, 'device_identification_results.csv'), index=False)

print('\nAll models and results saved to:', OUTPUT_PATH)
print('\nGenerated files:')
!ls -lh {OUTPUT_PATH}

In [ ]:
print('Done!')